# ДЗ 8 : Случайный лес

## 1. Разделение выборки и работа с признаками

In [41]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

df = pd.read_csv('ToyotaCorolla.csv')

X = df.drop(['Price', 'Id', 'Model'], axis=1)
y = df['Price']

cols_to_encode = ['Fuel_Type', 'Color']
encoder = OneHotEncoder(drop='first', sparse_output=False)
encoded_features = encoder.fit_transform(X[cols_to_encode])
column_names = encoder.get_feature_names_out(cols_to_encode)
encoded_df = pd.DataFrame(encoded_features, columns=column_names, index=X.index)
X = pd.concat([X, encoded_df], axis=1).drop(cols_to_encode, axis=1)

X.head()

,Age_08_04,Mfg_Month,Mfg_Year,KM,HP,Met_Color,Automatic,CC,Doors,Cylinders,Gears,Quarterly_Tax,Weight,Mfr_Guarantee,BOVAG_Guarantee,Guarantee_Period,ABS,Airbag_1,Airbag_2,Airco,Automatic_airco,Boardcomputer,CD_Player,Central_Lock,Powered_Windows,Power_Steering,Radio,Mistlamps,Sport_Model,Backseat_Divider,Metallic_Rim,Radio_cassette,Parking_Assistant,Tow_Bar,Fuel_Type_Diesel,Fuel_Type_Petrol,Color_Black,Color_Blue,Color_Green,Color_Grey,Color_Red,Color_Silver,Color_Violet,Color_White,Color_Yellow
0,23,10,2002,46986,90,1,0,2000,3,4,5,210,1165,0,1,3,1,1,1,0,0,1,0,1,1,1,0,0,0,1,0,0,0,0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,23,10,2002,72937,90,1,0,2000,3,4,5,210,1165,0,1,3,1,1,1,1,0,1,1,1,0,1,0,0,0,1,0,0,0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,24,9,2002,41711,90,1,0,2000,3,4,5,210,1165,1,1,3,1,1,1,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,26,7,2002,48000,90,0,0,2000,3,4,5,210,1165,1,1,3,1,1,1,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,30,3,2002,38500,90,0,0,2000,3,4,5,210,1170,1,1,3,1,1,1,1,0,1,0,1,1,1,0,1,0,1,0,0,0,0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Предобработка признаков для случайного леса и для линейной модели отличается, так как:
- для линейных моделей важно масштабирование, деревьям же неважно
- случайный лес устойчив к мультиколлинеарности, когда линейные модели - нет

In [42]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Размер обучающей выборки: {X_train.shape}")
print(f"Размер тестовой выборки: {X_test.shape}")

Размер обучающей выборки: (1148, 45)
Размер тестовой выборки: (288, 45)


Выборка делилась в отношении 80/20 тренировочной к тестовой.

## 2. Обучeние моделей, оценка качества и их сравнение

In [ ]:
dt = DecisionTreeRegressor(random_state=42)
rf = RandomForestRegressor(n_estimators=70, random_state=42)

def learn_model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')

    print(f"MAE:  {mean_absolute_error(y_test, y_pred):.2f}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")
    print(f"R2:   {r2_score(y_test, y_pred):.4f}")
    print(f"R2 (CV Mean): {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})\n")

learn_model(dt, X_train, y_train, X_test,  y_test)
learn_model(rf, X_train, y_train, X_test, y_test)

MAE:  981.63
RMSE: 1292.22
R2:   0.8749
R2 (CV Mean): 0.8158 (+/- 0.0697)



При кросс-валидации выброка делиться в среднем на 5-10 частей. Кросс-валидацию можно и не использовать и просто обучать модель на train/test.

### Сравнение моделей: Дерево решений и Случайный лес
Очевидно, случайный лес обучается дольше, чем дерево решений, так как содержит в себе множество этих моделей. Но качество случайного леса сильно лучше

### Метрики
Для сравнения использовались три метрики: MAE, RMSE, R2. Например, RMSE показывает на сколько численно ошибается модель, а R2 - показывает относительное значение, которое легко сранивать между моделями. 

### Выводы
Случайный лес справился лучше всего и имеет высокую точность.